# GwenLand glcuda Wave 133 - captured decode graph profile

A-B-B-A production control plus opt-in CUDA-event attribution inside the captured graph.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave133-captured-decode-profile-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "3e2622b7137f014f11b6892000b5c7be74b67c5f"
SOURCE_REV = "6007e3c69232dc1457c6253d518a1f5e5191a82a"
PATCH_SHA256 = "d2bc9239865e0687b7fe3b088d832834e2acd9a70ee8430107c432d876d0016b"
PATCH_GZIP_B64 = """H4sIAKGbuGoC/+19aXPbRrbod/+KjqbKISMS5r55nLmOrbhy4112MrcUFwQCTQkjEqAJULKu7ar3I94vfL/knaW70dhIKpPcuW9xpSIJ6D7d6D77OX06CBcL0W5fhKnwHlws/W3gPZCfvNV6KZMHN9617Pb67seJG8VJGm+ks0nE/LB298IokJ9ENxiMg8m4PxnMx70x/tf3O/5wPu8tup3O1Pen00G363UcR47m48lwOpf9oewuur2R7He90aQfSH/c8cajYW8yn3SnYwH9RoPBvXa7feic7x0fHx8873/7N9EejVsjcQz/Hwv4cxGJlRdGjaZofy/eymS7TP/aaLbED/Gnvwa3kUjSYDaTm028mc1O8Mf334vP94T+N5eRf7nyNlduIBdy4y4WkbuRSRhsveVMnMYr2Ug3W9lsVXXJ5qaaLrxlUtMWAV94qXS3azdJPf9KBoVOx+VOUXfkLraJDNzkJrxYbquG+dp8yL9crLdOGIVpo/k368ky9gJ3FQdy2bhPP/BtsBez+t3cwO56EwdbPw3jaAea7ex0L5I3YhEupcBpaERhROwc+M9xvFF/6HdH08V82g3GXt8LxuNhv7OAh+N5d9Sb9gfTnhxNCQMfBPL6QbRdLnfh2O5JI8J1Wh1x3G11x0NAuHvHDx58I36FngK6ijBqQwdfJonIOorHD34Qi3ijmvWm3yaCxhCnN+Gz5+/FldxEcukQLAb4Q5xeCm+zSuCdXIv0UoqNTAGxoZOC0hdepP6YTMTaSy8TISNvvpSBI95Bhzha3jI039sm3lLAF6X+pQgTcXMpAeTGnuN6I2E3ljDMeunBB9CYCjMFYuqD7ZqhPTt58UKsl9tET//j1ovS8D8BHgxwaX8lLK2ftpNLby3VFy+9LWAzfOq9Y/hbwBYgucxkdAHf5qYbL0xns8/Plif0oCV+ioAMf4rW2/TrQ9MFdo0a4S9P4mgRXrQE/8Xdsqa4GbPZM/rJ7x7i0H4cJal4/fbVi9fv3Pcvf3o3E/eTdCMeiaMX0ku2GwmfD+sUyFRuVkBFSRr6IrlNUrminV2tU1hVWDLApltHnHyCRQsjcRnfiDS+koAF3sZbLuUSMCKVF7A0Ky/dhJ/EChhSuF6GvkerDrNMBKzPBpZmJVfx5rYl3skoAWR5AguTtGiTF+Ens3TiQgLNpxsY9eih/pI37x+/fXfy7nQGAGEj4Du6HfPy18dvX7x/feq+Pnnrwq9Zm6FpcvL31ydP3p08ddWSvHv188lLC1pvMKB1A+4a4l40wiCBJTvb9nsfiNNm2wTclDhX+Qn+o8VxqTf8z0lj91r6DZvbrbxPLrAGl1pCs671DlZ/LTdeukUO23E69qt47V7NxKD4bI0Np0PrKSC49FJ3DbSyTG9hAMcewnGyic9mIAQ82LBGkxt8vXf8VS3DSgahFzVW8HmfYJa/SP+vi9Hge1oM+EV/86fEgb1M3flt44vXEvMvwnMANdLQW7r+at2YN51tdLPx1o0m8G3V5Qx6LSXKsAei9yEbc7ONXOAJDW7HJAO7YGO++hJre9QTH/AoDICMZ2Iex0v1NN54/hIeQUN4YstM+IZ9QtOahZNI+MZaOdUwo6O0oX5LmYoYFu+RhhDisjcMdjVNy3CBDR0mOoUW4ptHNTgrvnyh5gbRHC9xE6A3CcsJvc74kz/YSAmcdbuJBHxZA7g0EOo3jewl/jviTmIVJvDWv5wha370+avITQof6N9mf/sKzG8t/VQGj84+f/1w1MqDhK8yiyI+iyPzx5GAniDP8aFm+fCs0L20IhXvzRIU39HHWA+bFvoDu4ozXPzKP15dNXhAEhHuKmlahHB3bcsgAFKPt7lIAAu4ZXQ9m+GDRtNJrsJ1o9u00IXUFWiKDawZR/KTIVB6EF+58aZxBCLvAlB7j1AXL149PXkuzrwv8y/wFBWSD0c5LCX1hEdVY1kvN/JabhJ8T4ghk28a2B6RLpCwXLgYpKMdzY+adkc1lhtH/iG9VXOGkYOCkuiRLcocZnCNSW7tYKVB/YOWNreYzVBguz6J0IYtT23ySKTRTQe9asV0n7J8fICyXN+0Xleu77NHVT7WqvJxSVU+3qkqmxUFuoLVxHYwJrx1/UtY9Pu8IzbzQh7E3Lyead2JGWn1I45AXfJTEWzCRToTwH1ggM9fS5xGj194Xj2VA7lCHrN8hVmZljWbgQw3lOKX1tffu77MpVyWEQwgExD3cxIiaw5Nv8s6Wh9geKH1bBFukkrOwWqjBnkhI9Q5QAGLYubvR7nPSC63aRDfRI2MNFHXz7g76IVnhHctgfiaEz3Y0sUWYMsU9DS7GSGGkv73AedaAleglQ3SUrM1M7O4t9kuQMYc27EGYGai2TssY9VoOPvKgbB3KpeS9FKmivzcHfPWXm96A1/ggqkRNb6kX4SRMcVWameMnaWtlWzUbeRde+ES7Z+j3OTWG8DeZVQiozMlGdqa7YvPn387ygnV345mn7+2fju6BEblZsuDj2fOFN/Al+5/ka7VmxG+UfhJI9AAXw8m2WykwguzDIDoqVfRIGNCXkL66XdocYMKDSrm3s5lheFhHoOBKV8Qnmeg6FFSxOGavcjvB3Xl3Yi8lYQ1+u3o89ffjmDt9ASthfbBzDIbNb9NZQIyyAv0k5XnJzWrTLINx3JwmNqXNYuSNaAZ1L7NpqQ0fUTkTrO2PU54V8tmmcC1wMpxooI8ARWuhoFHKGVQr7KZlP4/PfpQVngCbG81MT+Iw+X4Mmgy0Bh4j9aVPmdAjKqrJvHVFrDRdays5EJ30KUsJdk7ynVDeWRcJY/QMNNajrf2/DC9bWhjGWigV9SRMq59cFckgI9bsOlozsjITasCo2+s4yTED2qh/eBmRhH2o4UiQeuGKQi6JugV2xVJHqDaAh0dxK5zY1TybQ0KWvIa57oUByUqzeyU/Eu9HxVdjBVT6PHw9/AGctUxc8gQxGYRais0/esl13/DF9qtK9l9JUP/3cyc2K+ZasVLNeOKNwZfyq/gO6rGqREcO4XHPvmws2NJNhR5lFI89mKWeemst8llw7I1Hx6KaRrR9kKoUpEyBsdAXPbwoG3Grh4D3V/GkWzkrDkz+VKv7LMK3SrQ3ULz7QqsmNsD8NwgLVNG4q7BEGMkp8eFzynKzvy0829N1zzKZ53yz5M12Inb9eHEUUkUmnfmHHZq4YuYXfg2603xw6xXNRZYGe3roR8MYsc0imi2s3GOyGyjtSTxWcyjd6YqppNs/AdgLYIczUVtco9VCCbojOej0RjDeN58MpzMx153OA8Gvcmo64/6k3Fv5C3Gge848+GoN5TTfjAZdIPBVA6Daacjvd6wOxl3JnKBUcLAmy4qY4H5oXNhmfwrivRNKNI3aYEBAH+HgPDi5FpG6dswurCjeF/v6R+8Kg8ePBBPvDX6jQV6B8R5eI7bBesusX97I32Qv6CFBFKAxMXQSHoJbz0RyLWEBYn82wyUDEBPE0/eP338bYJyO4yAVvOQYvJTo+4SJhQCiPAPUBRvYRDYkQxYGq5w9jR0sN3g70kK6uIKEIJm3BJJLC5AHbxsI6ANQNGdVtskxRBCBu38yXv35JeTl+/ctydPXr196p78/d3J25ePn587iu1s5xgj5Wm68hMDbNxP5HIB9iRGVsT9J/CjJULl/y86954t2ZVns2AOI9QNTg5mDEoUbEa1WI8Eju5469Dxty6to6smSOrXYuldJGwCusj7G1++FNm/mtNsxu6HxhHuDq9aeYth4wERbINROzgswVAycAkMOhhosvRX4lzItBE27zY37dbhOTLYebyNAsSiz+FXRBl0e8Sg83rRBTr9ds1LYcsj2jyHg0QuPyQXS+MVapaALrPZW7n0PsnAloeAM6ePfzx59x8zcU5TORdzCZLqIhFpzEGwJdCgQAAtcc5wz3GKGB/EVzlYCmf1pGCpyAUu5reAmhS2Uy3OObKFQDQSCtznHDQY5dpbhoEA2eKHi9AnArq5xKgx0Kf5dmg3l0gQCnjgZGC2UeItSuqCfyn9qwpN877GxFaVeoEY1KBVaqmhW7U4X6WCHflb5lcE6VfA7R8RtRu1MIq6ZLPSuSM09b8AfSdMAHgU4IqkN1JGxPASce6d04Kfz3HlFykGaz1QLmHR0DSBx46BkwF8S8Yj9H4Juss5ho2BY4bEH1lMwMpvIxwB9jje0HYyzxYYG8Xg5v/6H/+TmHe30yP23e0M+63u0DBwxIoc725WMO/HyyWqClIzPqY+xgGQm/CtfrzCEDlgUQW5h1ESBhaX9AyiMItwxKslmqk89QT2Fn1LFHnPs3flP0NayKDNgVY5Whx6F+jTDn1BfyUUjOdVw5UhPtZOvSv6hAi9VusYOA+0tuY2T4ghSAzaS0v+tDGer78cPhrlCMcxaFlkkoIGmOTZPH2d5qgwqubzkc3aX61RNv3VSNLv887/vZzZZkgZb+TxomYFmr7bbKVeFVwoRhj5aR1vUuYrxBmfMffO1gls2tCRGZ6KswJP+UDIkGx9XyIPMPhcpwstFmFJEeJnSgvqjbv+cDFejHqdSc/rz0dSdkbBxOt7/aDXWfS6XmfcGwznjtPvLhZyPBkN+/Bo4C8WQ9kNxvP5ZDyZ+r2FDLqgDnnjQa0WpMYtqUDqOZHQlFKd8Ee3hwSEuwx8aOun4ikt4uN1aNOS2lnFApnPiiNOYzgC/Gh8hz6PsN9DNrYAwcHOCXqQ4ceT9xsS/t+rDCMc1WCDDzSHQeXDhnryXnFP0Af2w2ZMOwC2AfvkPbPlImxDXQwKuARhWLfrdI9tDOPEmXOjNGkNBfGc82a81FLdiDdqJVK1BYKvVONQC4yxseenW5gAkWYGCxnVQ8FZJOc5MXFOiTwc7gmB18jEYgqki+IcAmtalxg7jGHIIIZ3yCeWEh3nXsYnmOmrcU6W3jqRwTt4aauJ9RQ/ywj+7ntz4OYnt6DHbOIIsPBwDNgPVvK3urgSs7tSyoIpxXxQ5ZhIqf1+Bym13x9xViJJukoSJaneyv9dIq7kduXG67SxDOctcf9srvWIJ/T+t87Rh1oQmobqQDCOKRDHO0DY278HmlFsds4st8F1AE+zRjuB5ba1FpqF6Duhgc6QbuLbekBPuYF73QM4LVF8roDXiR1O90serOKgJH7y75QYmncnwWg47wTzSWcxGsvFpDfEjMeunE79sT/sy8F8NPZGjjOcLLzhyPO6XRBBU3/oy+5iIjsdwFxvNJcduRh0R8GwVyuGCuOXxFHhPSJ7r9dpTcUx/uj2C2LpZ2p+KlON8+jhGk4NRf9Kf74ALWYpNb0qPg3WrsluvAH1rX2xAUvgZXcEjPYiBMLctNHFJzGlMfOyZaNMJrlRJhMzSrtqlG5vyimKrGeRxQG2x3+yg0pEGH3I2TPAZjNIVkLJZhtFpB/rrNCOk5M/h43oZDmlK+9WDSxCDDHkOH3FyMjaw1QlX4p1GGEkhHIwrdWBwXPLA3+b9THw9RRmOiu0KinUygclJTcGXf0G5R9nh2bQ6tNEE2gOJp1ZgSxPFBBoDdSTxivMv4xrPx0zbC89wLUkYR2QU10pgxaDK5HSKVGKAndXO1JOEnlMI/1AGXIGGX812bIcdE4u4+0yYL2VU3N7w5FowP/am/imyQsCNkcqvQCteMQUA2w0wEY4IwfMK2yAoXiwY9BWF4tNvBLnz57jZN23APH8Iebf2CqvIanZDMd1Vc7v+QeHqbHfbU2AGvvDVre7jxoRol7pbDK5STx7+9PT3tOd00DC7AX2RDgkSY91vmE7j1RdhZ5e8A/AKzDY/EtUfWA3YDsf6qVmykC7NSbCcYq4uQcM98qnOYMFmcLGqUni9mMuEux+PjXSwv8e4j8aY+ytwPH06NhNpW9rM5Naafsyg8U43iZnOChwmDKk/FNkCekPIE8axv3YSfVm0o48ni5B4o7FCYv8hHt9kFxX4Rr1FZH4G8wtEzebEFVHY4yi21hpqBqvCW8xUp1kEGkDsP2bCWqu4TV7Nf14fatW0E7kssknpweBxTJQxzNAUYqX17IiZ6xR3hJx/76V1BUDDwakklZ8XEbXIegHK1ICGAGb99h5YI1VSkit6KYWs2I4wyixAaf44m/aTL77/LKMXrMW9rY2KpBS7/xGftyCkqGnW5pMcfGK3XhsVkgr2cG///SO8Xw1lwFaM6/f/Z0oUM0VjR10h9hG0ytguCt3PDzOs7qUEtnbmOGPWZQgXNCGQj4HYNM4Fg3NZF6+cl+8ePyoe44US97PFooxxqHBEJTo4/6oY+nSpamz/pZ5r7OnZTyr2VjVx3Z3m2FyXu+2Bn0ql4vZjPLYSDsuY0LS8MmXvmMSLYHONf6/5d/7w2HrEYx3zezTc5Q8uOfMyESyBVbBQtqMypzYDE1KBp85ISccRiJ4t4bko+iPe/t26677kuc1+9tWZI3WdjKrXmQU/yyq5NIsFA1oT71Kf4RHDaAxc0rL+PXRp09bjKmPsQMUtvL+EW84SpM9C6N403xIiz/oTnHxByj7O7tXn3L3TQz66IxV/A+i136qTrCQup0TDYp/HTUflkJtetI59mNndV97GzdOGkeK4H98f3rivpm4oCCeHDWdMHETTM9tFmIbeb6IeS7VDDNvudqTKBi1tRN6evLk1dPKKeXz3nJfm6EjTM0eNb/OwIYrEPdeMTogMrFhQkm184WJvnx1+u7V29xcNR4MB+ghPB6MB8pTWI8IJp0iNxVkFffsVJ22/d1FKsGUT7lakZjVNll+I0t0ZQRSZd/i8h24tuV+ZR5QalO7xGBqEp4+dU9//QmwIo+nuQVRFhUnUZVWp5Bnu7OxtTCFLdnLQdzTF+Oh++vjX04Aeo6d2NSZ7VkZARkWBjVd7Q3GbGiXNmi18nAL5pSmWNKoXMt4o/xbwsPJBG2RYQd/7GVHrnj0PR73stGugldlK885M8y52jodnnNmSEt32QzQqTAZnugnFeihX5Wlk35z8dFzLzYxZ7rQA7JzTMIN2GT6dzB3l7DVE5OeSmtn+im0z//d7xXfG7IwYAtrb3Lb0hSmG98kLkhnP/sWmPHYJcMoyTWFHR2UHsD3XnwsP/Wuswyedi2ztWixVUV8VcpKS5gVbSmbsUUWdUvo9QOlnFauZViF/g0dsqV1apWoCgBmy0ID6gVpCf68VKVoqL9xEWoeXiMhmFfFpPKDsTEvwf4/jv7BOFqvEBSUiv+2OJtJgj8Re0WValOlFwa3kQemfVudVdZ6oYFUqRy+umrY3J5EwnSMzuJhV3sEdssEdhe3yg9BspQfgtwr7PwuDx+fUap1EZTTN3bs0AGNjU1WSOkoRiEIjfKPFIblH+7QcgsYnu9X7a/hxcjeFWdVeQav2L2iURFORjeEDuMOmYtjoIl9iiqlFpQVwVyeiPbUascd+c6175o9d9r1hvYEExFbuui5yJIwlugGu8VT/GF6205ZIeMKBm8mxuGnAr3sxlMuv3wCRn4ntAeI0zBKvhvzlfleFRkU+e/s9TlGUOPyQ1P9zYQNeE23xsvoZDFSmK5lptRONb8hVj2WmmQz3mewTMewz8NOnSZoJR29OH0Zb1YqkmCCLQJP0oBgXa29NISpFTySJrrQov3MwHkROpU46o/lOQgXNCMXXgDwOVqOY6ArHHPuApwFtNKlPzL39TtMWQmTnId6lnfo6goN5Daz81cwdHLL1RwsT/ZT7otb6C2TWLCnOUxUV/LYaoTWNTVwQwmF0QeTyx/SIaqHFAMCM47CDap2xEauCOM5nRRmk4a+woC/nHm43A1/Ga7Xt7NZGsfuyotuXW9zsUX3ZdL8kMOVzSpx9aIjqqKMatzLZ9K1VN5Kr4+7350Oh8D892y/xm3cjhB4Mx29NAEzZQH5MSVqqbSrDcbGkPHkwlxqniVGnMPslrgMg0BGbhCuuA4CIIn1R5TGVzOTsaBIoJ0jASV5MuMQrcqyfYQza2SD6XF4iGaBA5Rg5hk8mq3YrDhKLQ0We995emWvod6nKI7aixCpSzOXpXeLm4MITaoTbmW4ET/++DJPd5gRoxBk2OfMpsnYzg2sVQsc4jiNkkhge559fUoQtURV/mtZdJxyxJC4pI57mjo9OtRJ1KYTK60EBivOY8WajPNU4IkuYlMJZxgpLKb87UscbI4AlWd8fpsPc6qzcVYkITv7bVgX0IGKeWL4k1EchlVs5i9nQeyr3QUytiXUnsodikhyMZJmSWAVuxVO71q7VUksCnr1VlXkD8Ki0Bb1xlZmAHNL2i4lMDixEHP0fX+79jBdCvS1zW2eP+CMemP6ctQFt6BkuyzgLdmnnL+MkN+XpCAo1Vi0gUo2OCtv3fjCO/xFu1QyrwnLw36nT67afmfSa/VI84GWAhWNxIbuJYncpN80vtkRNcsd28zKIjRLaP6XM4Svtl99eak0hlZT0TKBsa9lgsezrYoZrimKY6NBaaIlTFLTpCCE7efVPWs78ofV9vtmT0er7sPvGTe/oPnjs9UL2it4p8lMABnFC2n7AlxNx3vWshgetE7c7l6S6o78NXfoZx303bmEld1yo5XoGKOL4QqPjxMzBAhLjHvTKqnIc06DMiIhA6NGp1Rz4H5USIWLYhhWjWCcdfqJC87xwQPgmEycGSRUHPHkRSRlgIq9J569fu/U5nEtw3kpf4uf6YKK/WlvOByMF/PO0O/1Op3RdNjtjrqTYW/qdQbeeNGZ9wcTzNsaTCf9oDvpL+QwGC/G3ng6GkjZGQy6U+jU6QyDSX8+8WvzttS4pXwt9ZyOTw3o+NSAVTArLaRQ+MUsxw/VwT+7lF3/28TkA6F8twWlUSxzssew3j31XSoCdIfNicS2JYVzlfZ4Xjtl4q4aMoUYIOlDubTitdyoFA88qzKHbwFMQ1sSI96gLGwvLsXZeb4Gz2Yb6eSd7oDyJbqDfqub2Ul281x6Nx/2q9BlOFssVzXQWp+7KzWZ/kJVHlC2AoX4pJpXKjwqF+tPUj/2n0fDaIe20vl4WUElUS+LxUiUDK+qPmJicqWzZkxulGiNVW50FEQfKsud2FLj7iuYplNFrJ76aGeJjZ5Q/SFxjolt53RqixAQCxRpz4ZWLanMzrcJt2BjGRRxYHVW0ga0/uHVKTdxxGN9Rk4dGkLk1YibFfA5//CQsW0bYQZuBo11V2TkasREbNc6DxwPziwA19K2xANLZC2rWpycVXAZLwNOahv0kC56w7FKalN0oWgCib+OSMIFIQOJcWzRBHzgU15+4eSRTiFx5hKAKB3Q1RVzboClx1TSqOocuT0K1zIyw3DZss+lE/HYHvYU88pND/oFjDj/qlzzQQXo4D0qZqkWtmp+jZ2H3MvW22tgVFy2JXMJJWlLFaEUc9ihK0wRy47pqeQQ4haph8jBWzOZtrqYn9OZtvrju+0N/nt96SXyNX+E+MxFULDWkKq6YicYUGnbqgQBXDknvx4FdXz9paqEBg/G1RtP8fd3dLzsewywYuEN4s2RBFJX6QGn7x4/O3FfPn5xclreHIfrdlS8sCp5VLyF6WLHL42wJdzmF7HmijJn34UfxPeiU9WFvgnbY+Ua6GJNvgptyB/tYWZ84zvqgYU3gQrxhFbVOUUunskbMIP5rJKz8INdlMbp1HWjqc/MN4Qfatplfi/rkC5IlwDDCYtbOqnSIpYKCvJiEfqOeI6HWGqhAYomUp+es89whxGC5tOhoBq2L8O07QX/2HI2XA009DUCRWxxToDAXspnb+i4JKVsK2p48fgJ8Kqb0JdONays/M+Ms7Cqm2HVn9oGX6tQwI+XKORLdE+1uDTxPFIorlCT8YYefSlUN2o6yXZVArafOI9t4jwu5DsxL0HivN54K9cwFT0TZpUurVFLXF27tEP6b+XG5j+bX4qM4wVBf2cKgH0Wh0MrcRV72uwrlVpxIDmRsRNTr4yPHualBDEoG48eIz76yrhBLMCyA0eI1iuuMxwc6bMFWI9AgEiOJeZa4llIJ3M0Uuk24tPoOYmokgh6TvRc80954a1nn3OSqhISs8/fCb1YhgUrTiEyVyeKkZRkaWBtXz6tQM1RR2sVbQjUqEFZ0H/qCbZM8eRVLCvpqADvbpAqDyeTG6fL+c1DOzf1YNHH2EVml7OrlmbrkG5WFPuQ5lWB7kKqXGW/UmaFFUIuZjCGqAOZZMWHOcJ4FdFBkk0KBsiSKn0vwytdsRs01LMkXAUf6NWMZBfpIGBWBDfeRtaa5EpBKhjl+qkyyxe9jrfw5cADk9yfDyZjLFDSCaZyPu773qAzWXQk/Js4Tn808TvBqN/1vGDuD8aTYWfkye6ED1NNg8l83pv0p9Nas9yMXDLMzRvEo2FniHiEP0pHqJ6tty+UCmm0N6X2gAhbq9zsNTrdAz77SZq6ow/to0q+NAcSHuZTtLXyR1od8yE6tY+eY22pN0gXarKgNjawUpDYxzqb0WHtk0/St431rMBFkp2415bmMr7Agg7Zt1DBQIx2WeVMVCre67evfvzp+Ylrq2Gnj7rn7E1XI6giAKxPqDof1ediTU39BBGL4nGgEgQbPD3ES5l52L3E0n6t4/V6VfJqJx8gTOpWKTtjb63SY9/frrYc5yrUeWENnYsecCCXDpdsNxvUcky1xQwWWykOy500O+2hDsLYRQzwjFpWpWDfhxW/yCjG1EqpCPYJvmoDg88WpRQCIYlIZ17Oda2uOXnvgspaFDl09gxU0FDQ7GRT15RnEKa2i0CSBBIJrST1aYdKUUy7QysaXaQzY7FfFzNNklJ6BhFGhdSpQY69LUuCxy7gtqvNfLsA6THDn4U3rLOxHoYq0Iyf1J9S5R0u8VLzWBeK6k8W8053NB55Xm/S6c3no1G3F3ij8dyfyLk3BQbaGw37jiP74wHwy85itJgMu4P+oL8YyMlkMpx0Fr0+8N+p9Oed4aiWmWZDl7hp9ops0m6X3AXwYzopsNPXvIpGqc0ckrxCWLh1lREwFvVXnr5jK1ivuBVaKUHVuS+R1R9yxA9c3CdUhYfmm9gLGJqMyCOYxpwZgAoi4SyXuEkk3iKRqoInmAogw4vLtM3+E3wMqPAP6XMeIsIz1EXBUET7LEHrJt5csVtlIWXALP6YzsZT7aayhTsTZ3gpxkPYjw9Y7pMRlvPtqAwzqK4kywExr8N4m2TR3WNrXJ3OcSzeTBSIj1fX1DdTB948+PnBL9bnJPZgAB8WQkIXtMxCD5Q09LNeojqNDqaWeBu/PmmJn39RKS9ZZ54DD4brwxk8P8NkkniRrrxP8NvjX+zBYvVd1N7KL3kzgabwEi+0sJedeqI2lS0J9TTfby0KesgrlsPSxbC/madx0FYNl02N1+U0fP5e3y5yW4SORlf2VfhXGehy5eKK6o3BWVBU35q/+vyI54/V++YgtzA4YwH7YN2wopDq+eP/OHmrpHZ2qUgZ41QJyjYVCstDeMHtDu7N9PAsL06VqqGKCHmW3I25ZJHWoqgcKN3H84ABYckA0GFQvyAaY0HKSoah+TjKlAp1tLKkPZgUIqtYEGsspUJxSmbrSkASrNmsThioKwlDM/VHqISc8tYpRQfIBKY250nwMFwoK55TbBfF/l/OMOZ+LRtP5Xx70RJPsE4leunt6jK2lNfWZFF3CYALoKlrF6GCCRhFjx0OmfcfueyZxWYf1u7pB0tberldzQFovOA0E6VjoGYEAl/9pWph2jts1zVhF9XZdjQ4bMgnMYIz6pnCE1IV0RUbmYIAlBbG7sdsOH2RDSFuS5395IIgHEgnva5yhSmBUAfb29/TmUDbli/+TZ4k/DSyvfd+W8Fnd9Y5vIv+KPu6na/5S3LUJ1Z/VrYzjfvoAecwD++bexfEUMlGpp5VIQZE3nXlacEzWhjCLhQvb6g0bR6cajGbedylHjP7/hN043N/XaEaTYkwAnlcrHjM4YHkTBU8f8Qg8k8LXldgu+V6tmypk79VgTqms0VMDI8eFZkozKmbFdBW2VrAqYAgSnO0Fg8D+dXFx2Es0+5zxdzUpT3H+QqQX0uXKlWpyy55YRuFbQZiYEGgb5Phj/iuSuLAmvXsI9dUzUYNpfmpvlMpb73d32e+5U4cc0kJ+SnlicNzxD5N9vuilgaFeA6IRsquVfuUK+Ru9knVoSzW8+STwd9lszFBSOuZvSFZ4djKrWCMLqxS7fK07E2Z5ZfgQMIuXhOkSOrRXt5mVWWn8VVNdoUgBfYw9+juHm5YiTx1zGIZpxarIP+6C/MENoGxsUYFpJ0MJBc/tDbf0YWNVkmDJnss1Azsv+BntzJe+B3O0zAW+P0QdrKzPDZoieS206uW7Fw2Xp2zAv/5UPl5GnIrGwM/KwdJVS8gQyyWKv+XraJvEzyk/As72Sl3OEpzCeIqAMC9G+c3Ceauf0zOH6hfE+CgMjlvtnQhS0ylt/pzmvHy9m8EQZ1eoIPWx+PpuDXs7fQk0HFhd013DOJ4V9dOAH8VPKZXqoWXNqxGLegZhKtinOJ6V2NYu6tr1Wv3keorp+aQQrEnkiHWnQMCz7mnAQb7+qvGUTzsUe5gQ94fYmIrFeMRs7JBAaRO7naLSm6eG5wPzLeYHZcAEmPMibLCChUFGiXNCzpm2++5j1++fPX+5ZOTpzM+SIz5M7PZK77Vq/AkfymS/leCRKKcbuipKCZcOAtbGUw0pxD5S9TFNl56SfXjVZJim+/m/MjnCKikPFcL5Q+Epijvv349Ko9RYhj5O0msCBixzKVS0phfIhow53Aq9apiJJa8QMaYnmNuBD7LR4uuHDyNgHapknyf1JgO2fT4QhHHp6glKK3dpy5ynVTeiAHUREHbBNAAswH4uAz+hYZ34z4Dv/no3FTEZu/fr+1wddcO19ChZn6Ypfi75gdYdbf57ehQMb8yCeEhfXtBK3AadvrHeBmoul7Kn0RjfJvwKYKNSjE0bIcKaqIEqISm+2a4w66ilGoYr8I0l2P6ZsKRehIflfDAwCMnFYccOK+JKpXZJ67oylhPeQLQGszSDcvoWjw8o+N3NVTtZ0pm8d+nmudV3Jr1k7r0jUqiqRk0qnmhJeuuzrbI3dUOSbVmOTT91rzvVjwvkXr9XSK2QqaXDtWyHUtal3lz5XhBkPGl7AwxSufijAoK2J/C4Arw/xgZKkqc++ODqwfXmT52Ckq6TTeWanX+KTpnF5z2n5WA6dNISZwpd1iEDElu5QVSldFDjzdeC3e5kZL0QXW7MqtrPao8fjzBNML+3sBPRg/AR3l8NKbxB+VGKIZZ9+Y6/6ZVBn7EK0S5ITRL9ZVZ2qY6ba05BZXmvzm6V5TG7eLNsrvEQfuu4qB9R3GQ71A2dHKyC9brm8ZucdGs2pgrx2KfmiYA9fPMp4LPaKorUFw+oCdX1+4Nxh0U5CuEBNiv15HHIn2+BGtf5yvufPW7Ol9z52vd+c8nZJsRzimrlacy/1i9LRmvo+UBQzVnlqCbqd/TBWImPcp7mAJBjg+iR9+JXEANDFQkClYFYeFrym8MivHR/5o1Q2WGIjgq4JDpIMTkHj9/LvgLGpSiFrhL0EkorsA+9KZTrt1DOIx3qNDNQg36RLWgoIE0neTjhpIBaVkH3VYP1xXrP1mVwHatbP2akdWDo/9L1pIpopoaYiJwHbbLSB7DUfQXwmvvkMV222p5/GfhB5o16kwKhmPbYCndeJvAMnCKPTBjijwcRlBmUcLj7Vq5QVTgKVEyVQnFEjBVGuCC76cFPfqso09CN0mKAkB+zg9bovedeu0canqhirRXMSnh+QVfvgg7o0KiRT+Jbggz1L4PuyXCz44xF4XjLmzKTgzjVC/UbfWFj03C5dZdbbWH9YKuVsQ+9qgWlzt8dM62Z4B/LA5XGmR1ElyvAgjyGjsN7R/D0doqJpxptWijsalF2pjR8iphZTWhUf3BMs1c3oADpo54pkLgerZ8PvvNpFUJTSmHNcYZ6hyUUT7HYgup3KiDOv9CA42kam6zd9pleYr6f9EsuzPu/wEsqso2/J3U84dorvvsxT/FvDvOA8V8vExXpfvAqHYJX9eFbj92l7eRAFkyJZh6GsO2XntAeTBoCWJ4Eek6LG8mmmyxBLxcY86CKvmuEk0wGcWkXVHuTQnefONh0ZOL8FomujcnPnp8VRSwqjZ1BY0KN64NnEYvUIEr7FPMq4XHn78Rd+PtKGN2M/ZCbhCXA9dVwGGXNe6ac58Pqe5QJTxKPMVklJswkW1yjCE6KLWHVBA83A2IsrytZMJa2OY48e9mwiRaq19hOao/nWlWaAiHMMM/EHVKK7IPqXfpJAdz7LuqTP+1X/9Pf/m/hMirCfhczfTcFHqiuxs5ZZo0IxhMRbuNRmXdKVJBv2xjaO3uONP7KAlQq2nWbZuUE1cJThcUUVWbsjusdNkFj2/5o9y8iBiHymeuYA41gciyQvXnqRZfc4ebcieOb4BduZj3RtVc9tpLFBRjRXq/OrLforDAGdWCJ1ONwofJEAtqjQj57xvWqF7g/3sV6J3M/a7o8X90kOROpHWYiLjz+lnBcV7Ku6xbccEw9YUL3mHhNRoXbGbpbznJhW4upev+ElX/Q4HRV+VGJXAmZ9k2iinPQer8ZVNFpCKguW/nqtbhD5boAWYuu7oaUeXBGpVqYg6tVga5K7JNHu1KSrSTGSj7rMh/VJVnlaANLVHMUWkiGWSHNTiHK0unP6q+caBY7oUKxuiKMXjvRnd6kKdXZRyghoM8GnX72kZFsxbmr2NajZ74LnMZkvZ2MJh8zIw7V5BfVrNedeQD7XZ5v2puAkNeObWFH3nSWgW3ptAS+mmzFm7xU7iyW+UNFhaNmfqV6uJe9OGyAYQqQmZUsTXKx3/WUm4qgT1595irKzK1U4Q1TB3xE1+ds6b7iPURALq5KsnK2xZgVdhn+thJsvTmFGTBzIgHHH9ASQ6f+7AWHtmPdFuIUTDZ6qf6XZgvqEoWqmukcRBRk8KBC5TiRl0A83H42ISpSkeUg8fl21ptpGup+NSmU41LVKD4Y8JlvllIN/kahjyW1cioxnrhskUAv2zXFThSLyp1fxwdfqjBKyCUggU78qAaL3pOf6YOh2Wlp+I1+mNevTypBKTOWOqSsCaa4F9uo6tvE6o13qQIBFek6tL1S93RYJir77KHwXwtP84VWjC6HFaY+KaRuhfbljh95z573ypqzJa2USD+uoU+dENL+XzdPyCfr5zb171Dbt9d8vwqc/6wsnxVxl9v2ubKJcwM21nZLsr9yyxxXVc/zP2FVWBrqutX/dNRI64gu7v9rgX8uuulcln9oE79cBnBEE9lLomVipfd3oRJJeFQQiWnKVq84Yaqsi4XOhBG9aAwk8x6f3MZLyWNtBMgF8065+U4f2jJAc0d58CBTW5xdmC9Bp6K25Uv5JOf1vbNpBkfdurhUakYWqtHdWLNlD1h+I2ji61Hd1x7c2AcR7t2hxjudg0cz40XiwQvxsJaE8j41APgvbsxQ1VNxTmqLkm98G7ux0rzFWYbqOwx4gyiyoOfMeN7GV5EfI9BLbwr54AbcfYQ7Q671xyywr1xdhuc+bb7Dc9K2OJY5DbqjkNxd3tn90BgObi3DQ+xp50S6fsbHQQtj1272+5049r/DoBXaX3mLcay7KsdkLCzUI+9Hgd3NrlSJdwHwxGpAePuqDX987QAuUxJDTj96flOReAqV/Y7O7LwR2gEtrquLoIQ3nq9DLMyolSqcIXaKd4H4XacfZu3Y761S3+3Pd8RMdFQjCNuH59UiMEbP6U7vLrjUeeAYhn2v84O1LJpZxcL4JhNew8jaf8zTGQvA7GZx665ort4R4to9wfrlLjuaDrmWvjDaau3e8Vtv4crP37TWMeJcnldXTuqTIwLDxtNm+qoAZaQJHOBCxfoy0TpEZgoMWfnZKuChUatQ++UtCo/hYmpQa+OwT9UFUVRC1kp/cczR9xz8PCKbMRSOj1+Q3dqq1LZ0INPkKuOSG94CMCoYFix5+njZzlwxvgjrTDewCzwpDxDwPoaCUZC4i1ZP6YyzmLpXaACAILLyR0DolXiCh41XiN9C0P1qaYdnepuHKwsc2RfO5iBqjiAbHmmxB3dVlUHiiu/Sl89yEeFGCiVkbTgV5wH2b1M6iKAf/6kU1vBZvuHa/mTBYRt1IAuV2txddUGlz6FbCAu6qItHm6eWUNYCmFFh/2zs1E1XKPucGpNc2vtDjhsVeti+N2LlZXpEo9f/wREYoonPcxVoDLFog45E5Y7RJor/fZUwpLrasQzIHU+P2zCgEiVNCWq3kFTaqw3sk3Pup1mDthmG7E5xJVDsc6EvsUYL+dCu50dHe35bVtfdX+qvN2XrRwscoLHeAVFi3/n6CMXljcXoaOfRflOLrGGFTpQLtUhB+bg486IOfi009qTfZtlqVBqv5OLIn6TkVrimj0pKw1EXcqlrvJIs5AFld//2x9HkjQnq0p4o1J1JehZkYf9xYrLvUGCecG1h66SosqYOwqfizDw+uNtYh1x3Ov0Oq3edN8GvKVyHk7+cC1LjUe2FCjbwva9X5ZJnF8rZpWMMQ36I7di+1fqoGWprAWfK7RRhGzqbpSKZhTKEBxwcreSJ/Eu2UdDCwd81QHyR6IkkAoNzeH/6qoEOl7EEJoPq29TU50Kpd3xrtt4g0IBdShkQY1czZLZTFVgKdaEtyqY8DSK45tbQqgiP51n90Qkb0wVIlP6lIv2Gf63Aa0uXGodShcRe2ffT2dwzq5DRFUN0UGkexIP4xaXoEGBAvswY2PIu6wL7/TXxFQV0F9Kz67VmBX2B+ay2nLRSVXyA2ViuXogVdqoKFNejXUV+5Q/VZ4veL+aS9Yx6WqzOL7S2eoe1bIxeSOiQRwa5b0+UrHAw5/QHc1jfbsA6bzqvi/YAFW442zR733YUbUjxy4ompUrmQpPlEI/maDldNzrTsY1pvNfzvzFRQPvksELMkoXAWH1p2S7xhIblkVc4RxrFbM2WuboLMWU56YQXUsQ9kUphn7opJWqk5zBr27gJmAIp3xbqZoB+dD4+lN3DVq2peUcOsc6VbVlx3jv9iG755/B3fUh+fJFrQq9TikQ+qZq3ClVFEVVYpw9ldeny9CX5RZUAXY2A0T4lRJkNbpM+a6DXqeDh3jqbobScMVngT44LMUkVGHHjl3xf8dFRf0CcbocoHaxxLXFENwVRozy2gAW4sKKxpxLpqoh8hlGL/iHR1U97U1Ktbn1/AWdNBLxTZQ3BtmuoztUvHDjiDc3Muo5w3bHGf5ART57A1NQpjf4bopVfYDkgKp0LZHivURkEe8wg3oDIGjo38zl25lKF4YNVcuDynIVd61SQ7yPa6R0TAGortMbNssNdhW8M31hteyu5jKEsrCCr79Lw8rF1b24vhM0LV8OlWvHNao6H1piMDmo6c5v3j/eigejlQFcau5vvmdAtT+d8n1WmsKETWE9zVlA2LgJUivo8ay8ATdyKYbkRt3ehDhkktfsrQlWxUGA3CfTUVPNCf/s0A1l/xvZ2hAzJrwAAA=="""
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q8_0.gguf?download=true"
MODEL_BYTES = 675710816
MODEL_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
ROOT = Path("/tmp/wave133")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
MODEL = ROOT / "qwen2.5-0.5b-instruct-q8_0.gguf"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave133-decode-graph-profile-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)

def run(cmd, *, cwd=None, env=None, timeout=14400, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       capture_output=True, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = sha256_file(FINAL_ZIP)
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest

def percentile(values, q):
    values = sorted(values)
    x = (len(values) - 1) * q
    lo, hi = math.floor(x), math.ceil(x)
    return values[lo] if lo == hi else values[lo] * (hi - x) + values[hi] * (x - lo)

def bootstrap_ci(values, seed=118, draws=20000):
    rng = random.Random(seed)
    n = len(values)
    medians = [statistics.median(values[rng.randrange(n)] for _ in range(n))
               for _ in range(draws)]
    return [percentile(medians, 0.025), percentile(medians, 0.975)]

phase = "bootstrap"
try:
    embedded = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    if hashlib.sha256(embedded).hexdigest() != PATCH_SHA256:
        raise RuntimeError("embedded patch hash mismatch")
    patch_path = RESULTS / "wave133.patch"
    patch_path.write_bytes(embedded)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(embedded),
    }, indent=2), encoding="utf-8")

    gpu = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff)

    cargo_candidates = [shutil.which("cargo"), Path.home() / ".cargo/bin/cargo",
                        "/usr/local/cargo/bin/cargo", "/opt/conda/bin/cargo"]
    cargo = next((str(x) for x in cargo_candidates if x and Path(x).is_file()), None)
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_script = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", rustup_script)
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {"CARGO_HOME": cargo_home, "RUSTUP_HOME": rustup_home}
        install = run(["bash", rustup_script, "-y", "--profile", "minimal",
                       "--default-toolchain", "stable", "--no-modify-path"],
                      env=cargo_env, timeout=1800)
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo, "bootstrapped": bootstrapped,
        "candidates": [str(x) for x in cargo_candidates if x],
    }, indent=2), encoding="utf-8")
    common = {**cargo_env, "CARGO_TARGET_DIR": TARGET, "CUDA_VISIBLE_DEVICES": "0"}
    candidate_switch_env = {
        **common,
        "GLCUDA_FUSE_Q8_GLUE": "1",
        "GLCUDA_DECODE_Q8_GLUE": "1",
    }

    phase = "host-tests"
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    if "75 passed" not in tests.stdout or "0 failed" not in tests.stdout:
        raise RuntimeError("unexpected host test summary")

    phase = "cuda-parity"
    parity = run([cargo, "test", "--release", "-p", "glcuda", "--test", "parity",
                  "--locked", "--", "--nocapture", "--test-threads=1"],
                 cwd=TREE, env=common, check=False)
    save("cargo-cuda-parity.log", parity)
    if parity.returncode or "0 failed" not in parity.stdout:
        raise RuntimeError("CUDA parity failed")

    phase = "graph-replay"
    graph = run([cargo, "test", "--release", "-p", "glcuda", "--test", "graph_replay",
                 "--locked", "--", "--nocapture", "--test-threads=1"],
                cwd=TREE, env=candidate_switch_env, check=False)
    save("cargo-graph-replay.log", graph)
    if graph.returncode or "4 passed" not in graph.stdout or "0 failed" not in graph.stdout:
        raise RuntimeError("Wave 132 graph replay gate failed")

    phase = "instrumented-graph-replay"
    profile_graph_env = {**candidate_switch_env, "GLCUDA_PROFILE_DECODE_STAGES": "1"}
    profile_graph = run([cargo, "test", "--release", "-p", "glcuda", "--test", "graph_replay",
                         "--locked", "--", "--nocapture", "--test-threads=1"],
                        cwd=TREE, env=profile_graph_env, check=False)
    save("cargo-instrumented-graph-replay.log", profile_graph)
    profile_graph_log = profile_graph.stdout + "\n" + profile_graph.stderr
    if (profile_graph.returncode or "4 passed" not in profile_graph.stdout
            or '"source":"cuda_events_inside_captured_graph"' not in profile_graph_log):
        raise RuntimeError("Wave 133 instrumented graph replay gate failed")

    phase = "compiler-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    resource = run([
        ptxas, "-v", "-arch=sm_75",
        TREE / "glcuda/src/kernels/glcuda_sm75_wave129.ptx",
        "-o", ROOT / "wave131.cubin",
    ], check=False)
    save("ptxas-wave129.log", resource)
    resource_log = resource.stdout + "\n" + resource.stderr
    expected_resource = [
        "Used 80 registers", "used 1 barriers", "16384 bytes smem",
        "0 bytes stack frame", "0 bytes spill stores", "0 bytes spill loads",
    ]
    if resource.returncode or any(item not in resource_log for item in expected_resource):
        raise RuntimeError(f"Wave 131 resource gate failed: {resource_log[-4000:]}")

    phase = "model"
    urllib.request.urlretrieve(MODEL_URL, MODEL)
    model_meta = {"bytes": MODEL.stat().st_size, "sha256": sha256_file(MODEL)}
    if model_meta != {"bytes": MODEL_BYTES, "sha256": MODEL_SHA256}:
        raise RuntimeError(f"model identity mismatch: {model_meta}")
    (RESULTS / "model.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")

    phase = "glbench-ceiling"
    glbench_build = run([cargo, "build", "--release", "-p", "glbench", "--locked"],
                        cwd=TREE, env=common)
    save("glbench-build.log", glbench_build)
    glbench = TARGET / "release/glbench"
    ceiling_prompt = ("Measure this deterministic systems prompt carefully. Explain how "
                      "token-parallel integer matrix multiplication uses shared memory, "
                      "Tensor Cores, and fixed launch geometry. " * 8)
    prod_env = {
        **common,
        "GLCUDA_FORCE_Q8": "1",
        "GLCUDA_GRID2D": "1",
        "GLCUDA_FUSE_Q8_GLUE": "1",
        "GLCUDA_Q8_NOSTORE": "1",
        "GLCUDA_FFN_GATE_UP_STACKED": "1",
        "GLCUDA_DEFER_FFN_RESIDUAL": "1",
        "GLCUDA_NTILE128": "1",
        "GLCUDA_BSTAGE": "1",
        "GLCUDA_GEMM_N16": "1",
        "GLCUDA_GEMM_N16_PREFETCH": "1",
        "GLCUDA_N16_FUSED_SWIGLU": "1",
        "GLCUDA_ATTN_MMA4": "1",
        "GLCUDA_ATTN_MMA4_REGQ": "1",
        "GLCUDA_ATTN_MMA4_AV": "1",
        "GLCUDA_DECODE_Q8_GLUE": "1",
    }
    profile_env = {**prod_env, "GLCUDA_PROFILE_DECODE_STAGES": "1"}
    profile_runs = []
    expected_stages = [
        "attn_input", "qkv", "attn_prepare", "attention", "attn_out",
        "ffn_input", "ffn_gate_up", "ffn_activation", "ffn_down", "lm_head",
    ]
    for index, arm in enumerate(["control", "profile", "profile", "control"]):
        out = RESULTS / f"glbench-{index + 1}-{arm}.json"
        env = profile_env if arm == "profile" else prod_env
        measured = run([
            glbench, "run", "--engine", "glcuda", "--model", MODEL,
            "--prompt", ceiling_prompt, "--tokens", "64", "--cold-iters", "1",
            "--warmup", "2", "--iters", "10", "--temperature", "0", "--seed", "42",
            "--kind", "end_to_end", "--verify-against", "glproc", "--out", out,
        ], cwd=TREE, env=env, check=False)
        save(f"glbench-{index + 1}-{arm}.log", measured)
        if measured.returncode or not out.is_file():
            raise RuntimeError(f"glbench profile run failed: {index + 1} {arm}")
        dispatch_log = measured.stdout + "\n" + measured.stderr
        if ('"path":"wave129-fused-swiglu-production"' not in dispatch_log
                or '"path":"wave132-exact-q8-glue"' not in dispatch_log):
            raise RuntimeError(f"production dispatch missing: {index + 1} {arm}")
        profiler_dispatched = '"source":"cuda_events_inside_captured_graph"' in dispatch_log
        if profiler_dispatched != (arm == "profile"):
            raise RuntimeError(f"profile dispatch mismatch: {index + 1} {arm}")
        session = json.loads(out.read_text(encoding="utf-8"))
        validation_text = json.dumps(session.get("validation", {}))
        if (not session.get("validation", {}).get("passed")
                or "50/50 tokens match oracle" not in validation_text):
            raise RuntimeError(f"oracle gate failed: {index + 1} {arm}")
        decode_profile = session.get("telemetry", {}).get("decode")
        if arm == "profile":
            names = [stage["name"] for stage in decode_profile["stages"]] if decode_profile else []
            if names != expected_stages or decode_profile.get("total_ms", 0) <= 0:
                raise RuntimeError(f"decode profile contract failed: {index + 1} {names}")
        elif decode_profile is not None:
            raise RuntimeError(f"control unexpectedly carried decode telemetry: {index + 1}")
        profile_runs.append({
            "order": index + 1,
            "arm": arm,
            "prefill_tps": session["analysis"]["prefill_tps"],
            "decode_tps": session["analysis"]["decode_tps"],
            "decode_profile": decode_profile,
        })

    profile_stages = {}
    for run_record in profile_runs:
        profile = run_record["decode_profile"]
        if not profile:
            continue
        for stage in profile["stages"]:
            profile_stages.setdefault(stage["name"], []).append(stage["total_ms"])
    ranked_stages = sorted(
        [
            {
                "name": name,
                "median_total_ms": statistics.median(values),
                "runs": len(values),
            }
            for name, values in profile_stages.items()
        ],
        key=lambda stage: -stage["median_total_ms"],
    )
    control_decode = [run["decode_tps"]["median"] for run in profile_runs
                      if run["arm"] == "control"]
    instrumented_decode = [run["decode_tps"]["median"] for run in profile_runs
                           if run["arm"] == "profile"]
    summary = {
        "wave": 133,
        "purpose": "captured_decode_stage_profile",
        "retention_authority": False,
        "gpu": fields,
        "model": model_meta,
        "runs": profile_runs,
        "control_decode_p50_tps": statistics.median(control_decode),
        "instrumented_decode_p50_tps": statistics.median(instrumented_decode),
        "instrumentation_slowdown": (
            statistics.median(control_decode) / statistics.median(instrumented_decode)
        ),
        "ranked_decode_stages": ranked_stages,
        "profile_source": "cuda_events_inside_captured_graph",
        "profile_is_perturbing": True,
    }
    report = [
        "# Wave 133 captured decode graph profile",
        "",
        "Diagnostic only: event nodes perturb the graph and are not retention authority.",
        f"- Control decode P50: {summary['control_decode_p50_tps']:.3f} tok/s",
        f"- Instrumented decode P50: {summary['instrumented_decode_p50_tps']:.3f} tok/s",
        f"- Instrumentation slowdown: {summary['instrumentation_slowdown']:.4f}x",
        "",
        "| stage | median total ms | runs |",
        "|---|---:|---:|",
    ]
    for stage in ranked_stages:
        report.append(
            f"| `{stage['name']}` | {stage['median_total_ms']:.6f} | {stage['runs']} |"
        )
    (RESULTS / "wave133-summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    (RESULTS / "REPORT.md").write_text("\n".join(report), encoding="utf-8")
    print("WAVE133_RESULT", json.dumps(summary, indent=2), flush=True)
    archive()
except Exception:
    (RESULTS / "FAILED.json").write_text(json.dumps({
        "phase": phase, "traceback": traceback.format_exc()}, indent=2), encoding="utf-8")
    archive()
    raise
